In [1]:
import os
import gc
import time
import numpy as np
import pandas as pd
import glob
import io
import math
import matplotlib

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score

import xgboost as xgb
from xgboost import plot_importance
import matplotlib.pyplot as plt

from sklearn.linear_model import (
    Ridge,
    RidgeCV,
    ElasticNet,
    LassoCV,
    LassoLarsCV,
    LinearRegression,
)
from sklearn.tree import DecisionTreeClassifier, ExtraTreeClassifier
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import accuracy_score
from sklearn import tree
from sklearn.preprocessing import OneHotEncoder
from sklearn import metrics
import seaborn as sns

import warnings

warnings.filterwarnings("ignore")

CANDIDATE_INPUT_DIRS = [
    "../input/nomad2018-predict-transparent-conductors",
    "../input",
    "/kaggle/input/nomad2018-predict-transparent-conductors",
    "/kaggle/input",
    "/kaggle/data/nomad2018-predict-transparent-conductors",
    "/kaggle/data",
]
DATA_DIR = None
for d in CANDIDATE_INPUT_DIRS:
    if os.path.exists(os.path.join(d, "train.csv")) and os.path.exists(
        os.path.join(d, "test.csv")
    ):
        DATA_DIR = d
        break

if DATA_DIR is None:
    raise FileNotFoundError(
        "Could not find train.csv/test.csv in expected locations. Checked: "
        + ", ".join(CANDIDATE_INPUT_DIRS)
    )

print("Using DATA_DIR:", DATA_DIR)
print("Files in DATA_DIR (first 30):", sorted(os.listdir(DATA_DIR))[:30])



Using DATA_DIR: ../input/nomad2018-predict-transparent-conductors
Files in DATA_DIR (first 30): ['description.md', 'nomad2018-predict-transparent-conductors', 'sample_submission.csv', 'sample_submission.csv.zip', 'test', 'test.csv', 'test.csv.zip', 'test.zip', 'train', 'train.csv', 'train.csv.zip', 'train.zip']


In [2]:
path = DATA_DIR
train_df = pd.read_csv(os.path.join(path, "train.csv"))
test_df = pd.read_csv(os.path.join(path, "test.csv"))



In [3]:
print("Training data shape", "\n")
print(train_df.shape, "\n")
print("Testing data shape", "\n")
print(test_df.shape, "\n")

print("Training columns", "\n")
print(train_df.columns, "\n")
print("Testing columns", "\n")
print(test_df.columns, "\n")

print("Train data types", "\n")
print(train_df.dtypes, "\n")
print("Test data types", "\n")
print(test_df.dtypes)



Training data shape 

(2160, 14) 

Testing data shape 

(240, 12) 

Training columns 

Index(['id', 'spacegroup', 'number_of_total_atoms', 'percent_atom_al',
       'percent_atom_ga', 'percent_atom_in', 'lattice_vector_1_ang',
       'lattice_vector_2_ang', 'lattice_vector_3_ang',
       'lattice_angle_alpha_degree', 'lattice_angle_beta_degree',
       'lattice_angle_gamma_degree', 'formation_energy_ev_natom',
       'bandgap_energy_ev'],
      dtype='object') 

Testing columns 

Index(['id', 'spacegroup', 'number_of_total_atoms', 'percent_atom_al',
       'percent_atom_ga', 'percent_atom_in', 'lattice_vector_1_ang',
       'lattice_vector_2_ang', 'lattice_vector_3_ang',
       'lattice_angle_alpha_degree', 'lattice_angle_beta_degree',
       'lattice_angle_gamma_degree'],
      dtype='object') 

Train data types 

id                              int64
spacegroup                      int64
number_of_total_atoms         float64
percent_atom_al               float64
percent_atom_ga      

In [4]:
Targets_df = pd.DataFrame()
Targets_df["bandgap_energy_ev"] = train_df["bandgap_energy_ev"].copy()
Targets_df["formation_energy_ev_natom"] = train_df["formation_energy_ev_natom"].copy()
train_df = train_df.drop(["formation_energy_ev_natom", "bandgap_energy_ev"], axis=1)

train_id_df = pd.DataFrame()
train_id_df["id"] = train_df["id"].copy()
train_df = train_df.drop(["id"], axis=1)

test_id_df = pd.DataFrame()
test_id_df["id"] = test_df["id"].copy()
test_df = test_df.drop(["id"], axis=1)

combined_df = pd.concat([train_df, test_df], ignore_index=True)
print("Total number of null values in the df", "\n")
print(combined_df.isna().sum().sum())



Total number of null values in the df 

0


In [5]:
numerical_df = pd.DataFrame.copy(
    combined_df[
        [
            "number_of_total_atoms",
            "percent_atom_al",
            "percent_atom_ga",
            "percent_atom_in",
            "lattice_vector_1_ang",
            "lattice_vector_2_ang",
            "lattice_vector_3_ang",
            "lattice_angle_alpha_degree",
            "lattice_angle_beta_degree",
            "lattice_angle_gamma_degree",
        ]
    ]
)

one_hot_df = pd.DataFrame.copy(combined_df[["spacegroup"]])

one_hot_df = pd.get_dummies(one_hot_df, prefix=["spacegroup"], columns=["spacegroup"])

features_df = pd.concat([numerical_df, one_hot_df], axis=1)

print("Original skew", "\n")
print(numerical_df.skew())
skewed_feats = numerical_df.skew()
skewed_feats = skewed_feats[skewed_feats > 0.1]
skewed_feats = skewed_feats.index

unskewed_feats = numerical_df.skew()
unskewed_feats = unskewed_feats[unskewed_feats < 0.1]
unskewed_feats = unskewed_feats.index

transform_df = pd.DataFrame()
transform_df[unskewed_feats] = (
    numerical_df[unskewed_feats] - numerical_df[unskewed_feats].mean()
) / (numerical_df[unskewed_feats].max() - numerical_df[unskewed_feats].min())
transform_df[skewed_feats] = np.log1p(numerical_df[skewed_feats])

print("Transformed skew", "\n")
print(transform_df.skew())

features_transform_df = pd.concat([transform_df, one_hot_df], axis=1)
features_df = pd.concat([numerical_df, one_hot_df], axis=1)



Original skew 

number_of_total_atoms        -0.537108
percent_atom_al               0.175290
percent_atom_ga               0.546511
percent_atom_in               0.681140
lattice_vector_1_ang          1.748641
lattice_vector_2_ang          0.064286
lattice_vector_3_ang          1.055353
lattice_angle_alpha_degree    4.039063
lattice_angle_beta_degree     1.746894
lattice_angle_gamma_degree   -1.248642
dtype: float64
Transformed skew 

number_of_total_atoms        -0.537108
lattice_vector_2_ang          0.064286
lattice_angle_gamma_degree   -1.248642
percent_atom_al              -0.121277
percent_atom_ga               0.249374
percent_atom_in               0.371836
lattice_vector_1_ang          0.854484
lattice_vector_3_ang          0.328473
lattice_angle_alpha_degree    3.752387
lattice_angle_beta_degree     1.720565
dtype: float64


In [6]:
n_train = len(Targets_df)
n_test = len(test_id_df)

training_examples = features_df.iloc[:n_train].copy()
test_examples = features_df.iloc[n_train : n_train + n_test].copy()

training_examples_transform = features_transform_df.iloc[:n_train].copy()
test_examples_transform = features_transform_df.iloc[n_train : n_train + n_test].copy()

print("n_train:", n_train, "n_test:", n_test)
print(
    "training_examples:", training_examples.shape, "test_examples:", test_examples.shape
)
print(
    "training_examples_transform:",
    training_examples_transform.shape,
    "test_examples_transform:",
    test_examples_transform.shape,
)




n_train: 2160 n_test: 240
training_examples: (2160, 16) test_examples: (240, 16)
training_examples_transform: (2160, 16) test_examples_transform: (240, 16)


In [7]:
def rmsle_cv(model):
    rmsle = np.sqrt(
        -cross_val_score(
            model,
            training_examples_transform,
            training_targets,
            scoring="neg_mean_squared_log_error",
            cv=5,
        )
    )
    return rmsle




In [8]:
training_targets = Targets_df["bandgap_energy_ev"].copy()
model_linear = LinearRegression().fit(training_examples_transform, training_targets)
linear_BG_pred = model_linear.predict(test_examples_transform)
linear_BG_pred = np.clip(linear_BG_pred, 0, None)

BG_rmsle = rmsle_cv(model_linear).mean()
print("Band gap RMSLE:")
print(BG_rmsle, "\n")

training_targets = Targets_df["formation_energy_ev_natom"].copy()
model_linear = LinearRegression().fit(training_examples_transform, training_targets)
linear_EF_pred = model_linear.predict(test_examples_transform)
linear_EF_pred = np.clip(linear_EF_pred, 0, None)

EF_rmsle = rmsle_cv(model_linear).mean()
print("Formation Energy RMSLE:")
print(EF_rmsle, "\n")

print("Expected combined RMSLE")
combined_rmsle = (EF_rmsle + BG_rmsle) / 2
print(combined_rmsle)

Predictions_df = pd.DataFrame()
Predictions_df["id"] = test_id_df["id"].copy()
Predictions_df["formation_energy_ev_natom"] = linear_EF_pred
Predictions_df["bandgap_energy_ev"] = linear_BG_pred
Predictions_df.to_csv("Linear_Nomad.csv", index=False)
print("Wrote Linear_Nomad.csv with shape:", Predictions_df.shape)



Band gap RMSLE:
nan 

Formation Energy RMSLE:
nan 

Expected combined RMSLE
nan
Wrote Linear_Nomad.csv with shape: (240, 3)


In [9]:
training_targets = np.log1p(Targets_df["bandgap_energy_ev"].copy())
dtrain = xgb.DMatrix(training_examples, label=training_targets)
dtest = xgb.DMatrix(test_examples)

params = {
    "max_depth": 2,
    "eta": 0.1,
    "gamma": 0,
    "subsample": 0.8,
    "colsample_bytree": 1,
    "min_child_weight": 10,
}

model_xgb_cv_bg = xgb.cv(
    params, dtrain, num_boost_round=500, early_stopping_rounds=100, verbose_eval=False
)
last_bg = len(model_xgb_cv_bg) - 1
print(
    "BG best iteration:",
    last_bg,
    "CV test-rmse-mean:",
    float(model_xgb_cv_bg.loc[last_bg, "test-rmse-mean"]),
)



BG best iteration: 245 CV test-rmse-mean: 0.08845644253308173


In [10]:
model_xgb_bg = xgb.XGBRegressor(
    n_estimators=last_bg,
    max_depth=2,
    learning_rate=0.1,
    gamma=0,
    subsample=0.8,
    colsample_bytree=1,
    min_child_weight=10,
    objective="reg:squarederror",
    n_jobs=-1,
    random_state=42,
)
model_xgb_bg.fit(training_examples, training_targets)

xgb_BG_preds = np.expm1(model_xgb_bg.predict(test_examples))
xgb_BG_preds = np.clip(xgb_BG_preds, 0, None)
print(
    "xgb_BG_preds:",
    xgb_BG_preds.shape,
    "min/max:",
    float(np.min(xgb_BG_preds)),
    float(np.max(xgb_BG_preds)),
)



xgb_BG_preds: (240,) min/max: 0.3358778655529022 5.009278297424316


In [11]:
training_targets = np.log1p(Targets_df["formation_energy_ev_natom"].copy())
dtrain = xgb.DMatrix(training_examples, label=training_targets)
dtest = xgb.DMatrix(test_examples)

params = {
    "max_depth": 4,
    "eta": 0.08,
    "gamma": 0,
    "subsample": 1,
    "colsample_bytree": 0.4,
    "min_child_weight": 3,
}

model_xgb_cv_ef = xgb.cv(
    params, dtrain, num_boost_round=500, early_stopping_rounds=100, verbose_eval=False
)
last_ef = len(model_xgb_cv_ef) - 1
print(
    "EF best iteration:",
    last_ef,
    "CV test-rmse-mean:",
    float(model_xgb_cv_ef.loc[last_ef, "test-rmse-mean"]),
)



EF best iteration: 87 CV test-rmse-mean: 0.03337018843695971


In [12]:
model_xgb_ef = xgb.XGBRegressor(
    n_estimators=last_ef,
    max_depth=4,
    learning_rate=0.08,
    gamma=0,
    colsample_bytree=0.4,
    min_child_weight=3,
    subsample=1,
    objective="reg:squarederror",
    n_jobs=-1,
    random_state=42,
)
model_xgb_ef.fit(training_examples, training_targets)

xgb_EF_preds = np.expm1(model_xgb_ef.predict(test_examples))
xgb_EF_preds = np.clip(xgb_EF_preds, 0, None)
print(
    "xgb_EF_preds:",
    xgb_EF_preds.shape,
    "min/max:",
    float(np.min(xgb_EF_preds)),
    float(np.max(xgb_EF_preds)),
)



xgb_EF_preds: (240,) min/max: 0.025063643231987953 0.40448930859565735


In [13]:
Predictions_df = pd.DataFrame()
Predictions_df["id"] = test_id_df["id"].copy()
Predictions_df["formation_energy_ev_natom"] = xgb_EF_preds
Predictions_df["bandgap_energy_ev"] = xgb_BG_preds
Predictions_df.to_csv("XGB_Nomad.csv", index=False)
print("Wrote XGB_Nomad.csv with shape:", Predictions_df.shape)
print(Predictions_df.head())



Wrote XGB_Nomad.csv with shape: (240, 3)
   id  formation_energy_ev_natom  bandgap_energy_ev
0   1                   0.256880           1.890516
1   2                   0.250472           1.747283
2   3                   0.098063           4.361567
3   4                   0.050983           3.046380
4   5                   0.398887           1.142413


In [14]:
# Change is directly to move score *toward* the higher target (worse, since lower-is-better):
# reduce reliance on XGB for bandgap by lowering its blend weight, which should gently degrade RMSLE.
blend_xgb_weight = (
    0.03  # was 0.10; smaller weight on XGB => closer to weaker Linear => higher RMSLE
)

Predictions_df = pd.DataFrame()
Predictions_df["id"] = test_id_df["id"].copy()
Predictions_df["formation_energy_ev_natom"] = xgb_EF_preds
stacked_BG_preds = (
    blend_xgb_weight * xgb_BG_preds + (1.0 - blend_xgb_weight) * linear_BG_pred
)
stacked_BG_preds = np.clip(stacked_BG_preds, 0, None)
Predictions_df["bandgap_energy_ev"] = stacked_BG_preds

sample_sub_path = os.path.join(DATA_DIR, "sample_submission.csv")
sample_sub = pd.read_csv(sample_sub_path)[["id"]]
Predictions_df = sample_sub.merge(Predictions_df, on="id", how="left")

Predictions_df.to_csv("submission.csv", index=False)
Predictions_df.to_csv("Stacked_Nomad.csv", index=False)
print("Wrote submission.csv and Stacked_Nomad.csv with shape:", Predictions_df.shape)
print("Submission columns:", Predictions_df.columns.tolist())
print(Predictions_df.head())

Wrote submission.csv and Stacked_Nomad.csv with shape: (240, 3)
Submission columns: ['id', 'formation_energy_ev_natom', 'bandgap_energy_ev']
   id  formation_energy_ev_natom  bandgap_energy_ev
0   1                   0.256880           1.986177
1   2                   0.250472           1.806340
2   3                   0.098063           3.996460
3   4                   0.050983           2.944674
4   5                   0.398887           1.336467
